# One-Hot Encoding

## What is it?
One-hot encoding is a technique used to convert **categorical data** (textual labels) into a **numerical format** that machine learning algorithms can understand.

## How it works
Each unique category value is transformed into a new **binary column** (0 or 1). 
- `1` indicates the presence of that category.
- `0` indicates absence.

## Example
| Color   | Red | Blue | Green |
|---------|-----|------|-------|
| Red     | 1   | 0    | 0     |
| Blue    | 0   | 1    | 0     |
| Green   | 0   | 0    | 1     |

## Key Points
- **Use when**: Categorical data has **no ordinal relationship** (e.g., colors, countries, genders).
- **Caution**: Creates many new columns (curse of dimensionality) if the categorical variable has high cardinality.
- **In Pandas**: Use `pd.get_dummies()`.
- **In Scikit-learn**: Use `sklearn.preprocessing.OneHotEncoder`.

In [40]:
import pandas as pd
import numpy as np

In [41]:
# ==============================
# MODULE 1: IMPORTING DATA SET
# ==============================
df = pd.read_csv('../data/cars.csv')
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


#### Checking Unique values

In [42]:
uniq_val=df['brand'].nunique()
print(uniq_val)
df['brand'].value_counts()


32


brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Force               6
Land                6
Isuzu               5
Kia                 4
Ambassador          4
Daewoo              3
MG                  3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [43]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [44]:
df['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

# 1. OneHotEncoding Using pandas

In [45]:
pd.get_dummies(df, columns=['fuel', 'owner'], drop_first=True, dtype=np.int8)
# Without drop_first, we will have 3 columns for fuel and 2 columns for owner. With drop_first, we will have 2 columns for fuel and 1 column for owner. This is done to avoid the dummy variable trap.
# dtype=np.int8 is used to reduce the memory usage. By default, the dtype is np.uint8, which can store values from 0 to 255. Since we only have 0 and 1 in our dummy variables, we can use np.int8, which can store values from -128 to 127.


"""
NOTE: We don't use pandas get_dummies for encoding nominal data in production. We use sklearn's OneHotEncoder for that. 
This is because pandas get_dummies does not handle unseen categories in the test set. 
If we have a category in the test set that was not present in the training set, pandas get_dummies will throw an error. 
sklearn's OneHotEncoder handles this by ignoring unseen categories and encoding them as all zeros.

"""

"\nNOTE: We don't use pandas get_dummies for encoding nominal data in production. We use sklearn's OneHotEncoder for that. \nThis is because pandas get_dummies does not handle unseen categories in the test set. \nIf we have a category in the test set that was not present in the training set, pandas get_dummies will throw an error. \nsklearn's OneHotEncoder handles this by ignoring unseen categories and encoding them as all zeros.\n\n"

# 2. OneHotEncoding using Sklearn


In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,0:4], df.iloc[:,-1], test_size=0.2, random_state=42)
print(f"Shape of Training set X : {X_train.shape}")
print(f"Shape of Test set X : {X_test.shape}")

Shape of Training set X : (6502, 4)
Shape of Test set X : (1626, 4)


In [47]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int8, handle_unknown='ignore')
# Fit one train and transform both train and test 
ohe.fit(X_train[['fuel', 'owner']])

X_train_new = ohe.transform(X_train[['fuel', 'owner']]) 
X_test_new = ohe.transform(X_test[['fuel', 'owner']])

print("Shape of New X_train : ",X_train_new.shape)
print("Shape of New X_test : ",X_test_new.shape)


Shape of New X_train :  (6502, 7)
Shape of New X_test :  (1626, 7)


In [48]:
ohe_cols = ohe.get_feature_names_out(['fuel', 'owner']).tolist() #convert into list
base_cols = ['brand', 'km_driven']

X_train_df = pd.concat(
    [X_train[base_cols].reset_index(drop=True), pd.DataFrame(X_train_new, columns=ohe_cols)], axis=1
)

X_test_df = pd.concat(
    [X_test[base_cols].reset_index(drop=True), pd.DataFrame(X_test_new, columns=ohe_cols)], axis=1
)

print("X_train Shape : ",X_train_df.shape," |  X_test Shape : ",X_test_df.shape)
X_train_df.head()

X_train Shape :  (6502, 9)  |  X_test Shape :  (1626, 9)


,brand,km_driven,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Tata,2560,0,0,1,0,0,0,0
1,Honda,80000,0,0,1,0,1,0,0
2,Hyundai,150000,1,0,0,1,0,0,0
3,Maruti,120000,1,0,0,0,1,0,0
4,Maruti,25000,0,0,1,0,0,0,0


### OneHotEncoding with Top Categories

#### Using Pandas

In [49]:
brand_counts = df['brand'].value_counts()
df['brand'].nunique()
threshold = 100

replace = brand_counts[brand_counts <= threshold].index

pd.get_dummies(df['brand'].replace(replace, 'Others'), drop_first=True, dtype=np.int8)

,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Others,Renault,Skoda,Tata,Toyota,Volkswagen
0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,0,0,0,1,0,0,0,0,0,0,0,0
8124,0,0,0,1,0,0,0,0,0,0,0,0
8125,0,0,0,0,0,1,0,0,0,0,0,0
8126,0,0,0,0,0,0,0,0,0,1,0,0


#### Using Sklearn

In [51]:
# 1. Find rare brands on training data only 
threshold = 100
brand_counts = X_train['brand'].value_counts()
rare_brands = brand_counts[brand_counts <= threshold].index  # list brand counts under threshold 

# 2. collapse rare categories (Use same rare_brands for both sets )

X_train = X_train.copy()
X_test = X_test.copy()
X_train['ohe'] = X_train['brand'].replace(rare_brands, 'Others')
X_test['ohe'] = X_test['brand'].replace(rare_brands, 'Others')

# 3. OneHotEncode (fit on test transform both)
ohe  = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int8, handle_unknown='ignore')
category_cols = ['ohe', 'fuel', 'owner']
ohe.fit(X_train[category_cols])

X_train_ohe = ohe.transform(X_train[category_cols])
X_test_ohe = ohe.transform(X_test[category_cols])

# 4. Build DataFrames and concat with numeric cols

ohe_cols = ohe.get_feature_names_out(category_cols).tolist()
base_cols = ['km_driven']

X_train_df = pd.concat(
    [X_train[base_cols].reset_index(drop=True), pd.DataFrame(X_train_ohe, columns=ohe_cols)], axis=1
)

X_test_df = pd.concat(
    [X_test[base_cols].reset_index(drop=True), pd.DataFrame(X_test_ohe, columns=ohe_cols)], axis=1
)

print("X_train Shape : ",X_train_df.shape," |  X_test Shape : ",X_test_df.shape)
X_train_df.head()


X_train Shape :  (6502, 18)  |  X_test Shape :  (1626, 18)


,km_driven,ohe_Ford,ohe_Honda,ohe_Hyundai,ohe_Mahindra,ohe_Maruti,ohe_Others,ohe_Renault,ohe_Tata,ohe_Toyota,ohe_Volkswagen,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,2560,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
1,80000,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0
2,150000,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0
3,120000,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0
4,25000,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
